# Product Lookup — Semantic Search and RAG

1. **ChromaDB**
2. **Bi-Encoder embeddings**
3. **Cosine similarity**
4. **Cross-Encoder reranking**
5. **Claude Sonnet**


In [1]:
import os
import pandas as pd
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer, CrossEncoder
from sklearn.metrics.pairwise import cosine_similarity
from dotenv import load_dotenv

load_dotenv()


C:\Users\goswa\anaconda3\envs\meridian\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [5]:
df = pd.read_csv('./resources/ProductLookupData/product_catalog.csv')

print("Shape:", df.shape)
df.head()


Shape: (222, 10)


,product_id,sku,product_name,description,category,sub_category,brand,price,stock_status,avg_rating
0,P1001,SKU-ELE-1001,Voltix Signature Headphone,The Voltix Signature Headphone delivers reliab...,Electronics,Headphones,Voltix,13230.33,In Stock,4.0
1,P1002,SKU-ELE-1002,Pulseon Premium Headphone,The Pulseon Premium Headphone delivers reliabl...,Electronics,Headphones,Pulseon,10762.92,In Stock,3.0
2,P1003,SKU-ELE-1003,Nexbyte Active Headphone,The Nexbyte Active Headphone delivers reliable...,Electronics,Headphones,Nexbyte,12755.34,In Stock,4.4
3,P1004,SKU-ELE-1004,Voltix Max Headphone,The Voltix Max Headphone delivers reliable per...,Electronics,Headphones,Voltix,39964.06,In Stock,3.9
4,P1005,SKU-ELE-1005,Aurea Lite Headphone,The Aurea Lite Headphone delivers reliable per...,Electronics,Headphones,Aurea,21693.16,In Stock,4.8


In [6]:
def build_document(row):
    return f"""
Product Name: {row['product_name']}
SKU: {row['sku']}
Category: {row['category']}
Sub-category: {row['sub_category']}
Brand: {row['brand']}
Description: {row['description']}
Price: ₹{row['price']:.2f}
Average Rating: {row['avg_rating']}
Stock Status: {row['stock_status']}
""".strip()

df["search_text"] = df.apply(build_document, axis=1)
df[["product_id", "product_name", "search_text"]].head()


,product_id,product_name,search_text
0,P1001,Voltix Signature Headphone,Product Name: Voltix Signature Headphone\nSKU:...
1,P1002,Pulseon Premium Headphone,Product Name: Pulseon Premium Headphone\nSKU: ...
2,P1003,Nexbyte Active Headphone,Product Name: Nexbyte Active Headphone\nSKU: S...
3,P1004,Voltix Max Headphone,Product Name: Voltix Max Headphone\nSKU: SKU-E...
4,P1005,Aurea Lite Headphone,Product Name: Aurea Lite Headphone\nSKU: SKU-E...


In [7]:
# Bi-Encoder & Cross-Encoder

bi_encoder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

print("Models loaded successfully.")


C:\Users\goswa\anaconda3\envs\meridian\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\goswa\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4849.67it/s]
C:\Users\goswa\anacond

Models loaded successfully.


In [11]:
# Vector DB
# Using chromaDB to store metadata and enable hybrid search

chroma_client = chromadb.PersistentClient(path='./chroma_db')
try:
    chroma_client.delete_collection("meridian_products")
except Exception:
    pass

collection = chroma_client.get_or_create_collection(
    name="meridian_products",
    metadata={"description": "Meridian Commerce product catalog"}
)

print("Collection ready.")


Collection ready.


In [12]:
# Generating embeddings

documents = df["search_text"].tolist()
#Not implementing any explicit chunking strategy as the document is a csv file and each row is in itself complete information
embeddings = bi_encoder.encode(
    documents,
    normalize_embeddings=True
).tolist()

metadatas = []

for _, row in df.iterrows():
    metadatas.append({
        "product_id": str(row["product_id"]),
        "sku": str(row["sku"]),
        "product_name": str(row["product_name"]),
        "category": str(row["category"]),
        "sub_category": str(row["sub_category"]),
        "brand": str(row["brand"]),
        "price": float(row["price"]),
        "stock_status": str(row["stock_status"]),
        "avg_rating": float(row["avg_rating"])
    })

ids = df["product_id"].astype(str).tolist()

collection.add(
    ids=ids,
    documents=documents, #plain chunks / each row product desc
    embeddings=embeddings, #embeddings of respective chunk
    metadatas=metadatas
)

print("Products stored:", collection.count())


Products stored: 222


In [13]:
# Wide search Retrieval: top-k=15

def initial_retrieve(query, top_k=15):
    query_embedding = bi_encoder.encode(
        query,
        normalize_embeddings=True
    ).tolist()

    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=min(top_k, collection.count()),
        include=["documents", "embeddings", "metadatas"]
    )

    docs = results["documents"][0]
    embeds = results["embeddings"][0]
    metas = results["metadatas"][0]
    ids = results["ids"][0]

    scores = cosine_similarity(
        [query_embedding],
        embeds
    )[0]

    candidates = [] # stores initially retrieved chunks

    for product_id, document, metadata, score in zip(
        ids, docs, metas, scores
    ):
        candidates.append({
            "product_id": product_id,
            "document": document,
            "metadata": metadata,
            "bi_encoder_score": float(score)
        })

    return sorted(
        candidates,
        key=lambda x: x["bi_encoder_score"],
        reverse=True
    )


In [14]:
# Cross-Encoding / reranking / narrowing search

def rerank_results(query, candidates, top_n=5):
    pairs = [
        [query, item["document"]]
        for item in candidates
    ]

    cross_scores = cross_encoder.predict(pairs)

    # storing rerabked chunks
    reranked = []

    for item, score in zip(candidates, cross_scores):
        item = item.copy()
        item["cross_encoder_score"] = float(score)
        reranked.append(item)

    return sorted(
        reranked,
        key=lambda x: x["cross_encoder_score"],
        reverse=True
    )[:top_n] # top_n = top_k = 5


In [33]:
import anthropic

client = anthropic.Anthropic(
    api_key=os.getenv("ANTHROPIC_API_KEY")
)

#HyDE query optimization
def generate_hyde_document(query):
    
    prompt = f"""
You are helping improve semantic product search for an e-commerce product catalog.

A customer entered this search query:

"{query}"

Generate a hypothetical product description that would ideally match
what the customer is looking for.

Do not answer the customer.
Do not mention that this is hypothetical.
Do not recommend specific products.

Write a concise product-style description including likely relevant
features, use cases, and characteristics that would help semantic
search retrieve suitable products.

For the above situation, return only the hypothetical product description.


However, if searched for product is unidentifiable then do not guess the product and clearly quote just the product name
being looked for without any additional hypothetical description and just generate the product search query as a minimalistic search prompt
and return the same.
OR
If the customer query does not relate to any product search query, return the original customer query as it is without any modifications.
"""

    response = client.messages.create(
        model="claude-sonnet-4-5",
        max_tokens=200,
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return response.content[0].text

In [37]:
# TEST: data retrieval.

query = "I need headphones for travelling"
hyde_query = generate_hyde_document(query)
print(hyde_query)

initial_results = initial_retrieve(hyde_query, top_k=15)
final_results = rerank_results(hyde_query, initial_results, top_n=5)

result_table = pd.DataFrame([
    {
        "Product": r["metadata"]["product_name"],
        "SKU": r["metadata"]["sku"],
        "Category": r["metadata"]["category"],
        "Bi-Encoder Cosine": round(r["bi_encoder_score"], 4),
        "Cross-Encoder": round(r["cross_encoder_score"], 4)
    }
    for r in final_results
])

display(result_table)


Premium wireless noise-cancelling headphones designed for travelers. Features active noise cancellation to block airplane engine noise and ambient sounds. Compact, foldable design with included hard carrying case for easy packing. Long battery life of 30+ hours for extended flights and trips. Comfortable over-ear cushions for all-day wear during long journeys. Bluetooth connectivity with multi-device pairing. Built-in microphone for taking calls on the go. Lightweight construction ideal for portability. Quick charge capability for last-minute travel preparation. Includes airplane adapter for in-flight entertainment systems. Sweat and weather-resistant materials suitable for various travel conditions.


,Product,SKU,Category,Bi-Encoder Cosine,Cross-Encoder
0,Pulseon Premium Headphone,SKU-ELE-1002,Electronics,0.4265,0.1494
1,Voltix Max Headphone,SKU-ELE-1004,Electronics,0.3805,-1.1341
2,Nexbyte Active Headphone,SKU-ELE-1003,Electronics,0.3952,-1.4417
3,Aurea Classic Bluetooth Speaker,SKU-ELE-1019,Electronics,0.3846,-2.1949
4,Voltix Signature Headphone,SKU-ELE-1001,Electronics,0.3472,-2.3983


In [38]:
# TEST: LLM RAG answer generation

# RETRIEVED
context = "\n\n".join(
    [
        f"Product {i+1}:\n{item['document']}"
        for i, item in enumerate(final_results)
    ]
)
# AUGMENTATION
prompt = f"""
You are the Meridian Commerce product discovery assistant.

A customer searched for:
"{hyde_query}"

Below are products retrieved from the catalog and reranked for relevance:

{context}

Recommend the most relevant products based ONLY on the retrieved catalog context.

Instructions:
- Explain briefly why the best matches fit the customer's request.
- Mention product names and SKU values.
- Do not invent product specifications.
- If no product is a strong match, say so clearly.
- Keep the answer concise and customer-friendly.
- However, if searched for product is unidentifiable then do not guess the product and clearly state the same politely.
- If the customer search does not relate to any product search query, politely prompt the customer towards product search.
"""
#GENERATION
response = client.messages.create(
    model="claude-sonnet-4-5",
    max_tokens=500,
    messages=[
        {"role": "user", "content": prompt}
        ]
    )
print(response.content[0].text)


Based on your search for premium wireless noise-cancelling headphones for travel, here are the most relevant options from our catalog:

## Top Recommendations

**1. Pulseon Premium Headphone (SKU-ELE-1002)** - ₹10,762.92
This is the best match as it features **noise isolation** and is specifically **built for travel**. It comes with a solid 24-month warranty and is currently in stock. Rating: 3.0/5

**2. Voltix Signature Headphone (SKU-ELE-1001)** - ₹13,230.33
A good alternative with **long battery life** and also **built for travel**. This aligns with your requirement for extended usage during flights. It has a 4.0/5 rating and is in stock.

**3. Voltix Max Headphone (SKU-ELE-1004)** - ₹39,964.06
Features **noise isolation** and offers reliable performance with an 18-month warranty. While designed for daily commutes rather than travel specifically, it has the highest rating at 3.9/5.

## Important Note
While these headphones offer noise isolation and travel-friendly features, the cata